# Evaluative Framing of Scholars in Historical Texts


#### This notebook presents a computational approach to examining how historians evaluate and frame scholars and other historical figures in their writings.

The llm assisted analysis identifies evaluative terms and expressions associated with named individuals and classifies their framing as positive, negative, neutral, or mixed. The workflow includes scholar identification, LLM-based evaluative framing analysis, extraction of structured results and normalization of personal names.

The resulting dataset supports the comparative study of evaluative representations in historical narratives.

In [ ]:
import pandas as pd
import numpy as np
import re
import string

#to be able to track progress when applying functions
from tqdm.notebook import tqdm
from tqdm.notebook import tqdm_notebook
tqdm_notebook.pandas()


In [ ]:
# import the extracted quotes df
df = pd.read_csv('Kordatos_with_quotes.csv')

In [ ]:
df.head(5)

In [ ]:
print(df.columns.tolist())

In [ ]:
# use the monotonic version of the full text and the authorial text
df = df[['Chapters', 'text_monotonic_accents', 'text_No_Quotes']]
df

In [ ]:
import re
import pandas as pd
import json

# Define the scholar patterns using regex
scholar_patterns = [
    r'Αιλιαν\w*', r'Άμαντ\w*', r'Ανανί\w*', r'Αναγνωστόπουλ\w*', r'Αναγνωστόπούλ\w*',
     r'Ανθρακίτ\w*', r'Ασώπι\w*', r'Ασωπί\w*', r'Βαλαωρίτ\w*', r'Βάρναλης',
    r'Βάμβα\w*', r'Βαρβαγιάννη\w*', r'Βαρθόλδης', r'Βασιλειάδ\w*', r'Βασιλόπουλ\w*', r'Βελεστινλής', r'Βελεστινλή',
    r'Βελών', r'Βελάστ\w*', r'Βενιζέλ\w*', r'Βενιαμίν', r'Βερναρδάκ\w*', r'Βηλαρ\w*', r'Βικέλα\w*',
    r'Βιλαρ\w*', r'Βιλλουασών', r'Βλάχο', r'Βλάχος', r'Βλάχου', r'Βλαστ\w*',
    r'Βολταίρ\w*', r'Βουλγάρεως', r'Βούλγαρη\w*', r'Βούλγαρις',
    r'Βραγκοβάν\w*', r'Γαζή\w*', r'Γερλάχι\w*', r'Γερλαχί\w*', r'Γεωργιλά\w*', r'Γκυς', r'Γαβριηλίδη\w*',
    r'Γλην\w*', r'Γορδι\w*', r'Πατριάρχη\w*', r'Δαμωδ\w*', r'Δελμούζ\w*',
    r'Δοσίθε\w*', r'Δούκα', r'Δούκας', r'Δουκάγγι\w*',
    r'Ελλάδι\w*', r'Ελλαδί\w*', r'Έρασμ\w*', r'Εφταλιώτ\w*', r'Ζαμπέλ\w*',
    r'Ζήνο\w*', r'Ζυγομαλ\w*', r'Ηλιάδ\w*', r'Θεοτόκ\w*', r'Ιλαρί\w*',
    r'Ιλδεβράνδ\w*', r'Καβαλιώτ\w*', r'Καβάσιλ\w*',
    r'Κακριδή\w*', r'Καλλεμβέργ\w*', r'Καπνί\w*', r'Καινιγ\w*',
    r'Καταρτζ\w*', r'Κοδρικ\w*', r'Κομμητ\w*', r'Κονδυλάκη*',
    r'Κονταρ\w*', r'Κόντ\w*', r'Κούμα\w*', r'Κοραή\w*', r'Κορνάρ\w*', r'Κριτόπουλ\w*',
    r'Κρομμύδ\w*', r'Κρούσ\w*', r'Κυμηνίτ\w*', r'Κωνσταντά\w*',
    r'Λάγγι\w*', r'Λάνδ\w*', r'Λάσκαρ\w*', r'Λορεντζάτο\w*', r'Λασκαράτο\w*', r'Λουδόλφ\w*', r'Λούκαρ\w*',r'Λουκάρ\w*',
    r'Μάξιμ\w*', r'Μαλάκ\w*', r'Μαργούνι\w*', r'Μεθόδι\w*',
    r'Μερκάδ\w*', r'Μηνιάτ\w*', r'Μιστριώτη\w*',
    r'Μοισιόδα\w*', r'Ναουσαί\w*', r'Νερουλ\w*', r'Νικηφόρ\w*', r'Νιρβάνα\w*',
    r'Νοταρ\w*', r'Ξανθόπουλο\w*', r'Ξανθουκίδ\w*', r'Οικονόμ\w*', r'Οστοβήκ', r'Παλαμά\w*',
    r'Πάλλ\w*', r'Παπαμόσχου', r'Παπαντωνίου', r'Παρθένι\w*', r'Παρισιν\w*',r'Πάριο\w*',
    r'Πηγά', r'Πηγάς', r'Πιακεντίν\w*', r'Πιανζόλ\w*', r'Πολυλά\w*', r'Πόρκι\w*', r'Πουότ\w*', r'Ραγκαβή\w*',
    r'Ραρτούρ\w*', r'Ροΐδη\w*', r'Ρουσάν\w*', r'Ρωμαν\w*', r'Σκούφο\w*', r'Σολωμός', r'Σολωμού', r'Σούτσο\w*',
    r'Σομμαβέρ\w*', r'Σοφιανός', r'Σοφιανού', r'Σπων', r'Στρατηγ\w*',
    r'Στουδιτ\w*', r'Σύλβι\w*', r'Τριανταφυλλιδ\w*', r'Τρικούπη\w*', r'Τριββεχόβ\w*', r'Τσούντα\w*', r'Τυπάλδο\w*',
    r'Φαρμακίδ\w*', r'Φίλελφ\w*', r'Φιλήντα\w*', r'Φωτιάδ\w*', r'Φιλιππίδ\w*', r'Χατζηδάκ\w*',
    r'Χριστόπουλ\w*', r'Χρυσόστομος', r'Χύχ\w*', r'Χατζής', r'Χατζή', r'Ψαλλίδα\w*', r'Ψυχάρ\w*',     r'Ξενοφών\w*',
    r'Ξενοφώντ\w*', r'Πτωχοπρόδρομ\w*', r'Χορτάτζ\w*', r'Αθανασί\w*', r'Παπαδόπουλ\w*', r'Φιλαδελφ\w*', r'Γεωργί\w*', r'Βλαντ\w*',
    r'Πέζαρ\w*', r'Θερειαν\w*', r'Τερτσέτ\w*', r'Σπηλιωτάκ\w*', r'Λιβαδ\w*', r'Χρυσοβέργ\w*', r'Ζουφρέ\w*',
    r'Καλιάφ\w*', r'Λογοθέτ\w*', r'Δημητρακόπουλ\w*', r'Στελλάκ\w*',
]




In [ ]:
# Compile regex pattern
compiled_pattern = re.compile(r'\b(?:' + '|'.join(scholar_patterns) + r')\b')

def detect_scholars(text):
    """
    Detects scholars in a given text using predefined regex patterns.
    Returns a list of matched scholars.
    """
    matches = re.findall(compiled_pattern, text)
    return list(set(matches)) if matches else ["None"]  # Remove duplicates, return None if no match

In [ ]:
# Apply scholar detection on the 'text_no_quotes' column
df['Detected_Scholars'] = df['text_No_Quotes'].progress_apply(detect_scholars)

In [ ]:
# Flatten the list of scholars and get unique values
unique_scholars = set(scholar for scholars_list in df['Detected_Scholars'].dropna() for scholar in scholars_list)
print(unique_scholars)

In [ ]:
#display the texts better
pd.set_option('display.width', 50)
pd.set_option('display.max_colwidth', 50)

In [ ]:
df

### Μake test dfs

In [ ]:
test0_1 = df[0:2] # preface and Introduction
test1 = df[2:3] #chap 1
test2 = df[3:4] #chap 2
test3 = df[4:5] #....
test4 = df[5:6]
test5 = df[6:7]
test6 = df[7:8]
test7 = df[8:9]
test8 = df[9:10]
test9 = df[10:]

In [ ]:
pd.set_option('display.width', 50)
pd.set_option('display.max_colwidth', 150)

In [ ]:
test5

In [ ]:
#display the texts better
pd.set_option('display.width', 50)
pd.set_option('display.max_colwidth', 100)

# LLM impelentation for framing detection

In [ ]:
import json
from openai import OpenAI

client = OpenAI()  

In [ ]:
def analyze_framing(text, Detected_Scholars, model="gpt-4.1"):

    scholar_names = ", ".join(Detected_Scholars)  # Format scholar names for clarity    # Debugging: Print the scholar_names inside the function
     
    if not Detected_Scholars:
        return '{"logioi": []}'       
    
    prompt = f"""
    Είσαι  ένας αναλυτής λογου και αναλύεις πως ο συγγραφέας του δοθέντος κειμένου "{text}" αξιολογεί και πλαισιώνει συγκεκριμένους λογίους και άλλες προσωπικότητες που αναφέρει στο κείμενοt.

    Δοθέντος του ακόλουθου κειμένου:
    "{text}"

    1. Οι λογίοι που εμφανίζονται είναι μόνο αυτοί στη λίστα: {scholar_names}. Μην αναφέρεις άλλους λογίους που δεν περιλαμβάνονται στη λίστα.
    2. Προσδιόρισε τους αξιολογικούς όρους (θετικούς ή αρνητικούς) που χρησιμοποιούνται για να τους περιγράψουν.
    3. Προσδιόρισε φράσεις που χρησιμοποιούνται με αξιολογικό τρόπο (π.χ. 'επιμόνως εδόξαζεν', 'είχε την παράδοξη ιδέα')
    4. Αποφάσισε αν η πλαισίωση κάθε λογίου είναι «θετική», «αρνητική», «ουδέτερη» ή «μικτή». Χρησιμοποίησε «μικτή» όταν συνυπάρχουν σαφείς θετικές και αρνητικές αξιολογήσεις του ίδιου λογίου. Χρησιμοποίησε «ουδέτερη» όταν δεν διαπιστώνεται σαφής θετική ή αρνητική αξιολόγηση.
    5. Ταξινόμησε  την αξιολογική πλαισίωση του κάθε ατόμου στη λίστα, όχι απλά τις γνώμες ή τις στάσεις που εκφράζει το αναφερόμενο άτομο προς άλλα πρόσωπα. Να διακρίνεις ξεκάθαρα ανάμεσα στην αξιολόγηση του ίδιου του συγγραφέα του κειμένου και τις αξιολογήσεις που αφορούν τρίτα πρόσωπα.
    6. Δωσε με μια πρόταση την αιτιολόγηση για την απόφαση σου. 

    Επίστρεψε αποκλειστικά έγκυρο JSON με την ακόλουθη δομή:
    {{
  "logioi": [
    {{
      "onoma": "Όνομα λογίου",
      "mentions": [
        {{
          "axiologikoi_oroi": [],
          "fraseis": [],
          "plaisiosi": "θετική",
          "aitiologisi": "Σύντομη αιτιολόγηση."
        }}
      ]
    }}
  ]
}}

Οι επιτρεπόμενες τιμές του "plaisiosi" είναι:
"θετική", "αρνητική", "ουδέτερη", "μικτή".

Αν δεν εντοπίζεται κανένας από τους λογίους της λίστας, επιστρέψτε {{"logioi": []}}.
"""

    try:
        response = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "user", "content": prompt}
            ],
            temperature=0,
            response_format={"type": "json_object"}
        )

        json_output = response.choices[0].message.content
        return json_output

    except Exception as e:
        print(f"Σφάλμα κατά την κλήση του OpenAI API: {e}")
        return None

In [ ]:
from tqdm.auto import tqdm
tqdm.pandas()

test9["Framing_Analysis"] = test9.progress_apply(
    lambda row: analyze_framing(
        row["text_No_Quotes"],
        row["Detected_Scholars"]
    )
    if isinstance(row["Detected_Scholars"], list) and row["Detected_Scholars"]
    else '{"logioi": []}',
    axis=1
)

In [ ]:
# run evaluative framing analysis only for unprocessed chapters
# save a checkpoint after each iteration to allow resuming interrupted API processing without repeating completed calls 

# if there is no column 
if "Framing_Analysis" not in test9.columns:
    test9["Framing_Analysis"] = None

# go over all the rows of the chapter
for idx, row in tqdm(test9.iterrows(), total=len(test9)):

    if pd.notna(row["Framing_Analysis"]):
        continue

    scholars = row["Detected_Scholars"]

    if isinstance(scholars, list) and scholars:
        result = analyze_framing(
            row["text_No_Quotes"],
            scholars
        )
    else:
        result = '{"logioi": []}'

    test9.at[idx, "Framing_Analysis"] = result

    # create a checkpoint for each row
    test9.to_pickle("test9_framing_checkpoint.pkl")

In [ ]:
def make_framing_df(df):
    records = []

    for _, row in df.iterrows():

        if not isinstance(row["Framing_Analysis"], str):
            continue

        try:
            data = json.loads(row["Framing_Analysis"])
        except json.JSONDecodeError:
            continue

        for scholar in data.get("logioi", []):
            records.append({
                "Chapter": row["Chapters"],
                "Scholar": scholar["onoma"],
                "mentions": scholar.get("mentions", [])
            })

    if not records:
        return pd.DataFrame()

    framing_df = pd.json_normalize(
        records,
        record_path="mentions",
        meta=["Chapter", "Scholar"]
    )

    return framing_df.rename(columns={
        "axiologikoi_oroi": "Axiologikoi_Oroi",
        "fraseis": "Fraseis",
        "plaisiosi": "Plaisiosi",
        "aitiologisi": "Aitiologisi"
    })

In [ ]:
#display the texts better
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

### Manual checks per chapter

In [ ]:
framing_df0_1 = make_framing_df(test0_1)
framing_df0_1.tail(10)

In [ ]:
framing_df1 = make_framing_df(test1)
framing_df1.head(10)

In [ ]:
framing_df2= make_framing_df(test2)
framing_df2.head(10)

In [ ]:
framing_df3= make_framing_df(test3)
framing_df3.head(10)

In [ ]:
framing_df4= make_framing_df(test4)
framing_df4.tail(10)

In [ ]:
framing_df5= make_framing_df(test5)
framing_df5.tail(10)

In [ ]:
framing_df6= make_framing_df(test6)
framing_df6.tail(10)

In [ ]:
framing_df7= make_framing_df(test7)
framing_df7.head(10)

In [ ]:
framing_df8 = make_framing_df(test8)
framing_df8

In [ ]:
framing_df9 = make_framing_df(test9)
framing_df9.head(10)

### Concatenate

In [ ]:
combined_df = pd.concat([framing_df0_1, framing_df1, framing_df2, framing_df3, framing_df4, framing_df5, framing_df6, framing_df7, framing_df8, framing_df9], ignore_index=True)

In [ ]:
combined_df.head(4)

In [ ]:
combined_df.columns

In [ ]:
combined_df = combined_df [['Chapter', 'Scholar', 'Axiologikoi_Oroi', 'Fraseis',
       'Plaisiosi', 'Aitiologisi']]

### Normalize scholars names

In [ ]:
name_mapping = {
    "Φιλιππίδη": "Φιλιππίδης",
    "Γληνού": "Γληνός",
    "Γληνό": "Γληνός",
    "Κακριδή": "Κακριδής",
    "Κοδρικά": "Κοδρικάς",
    "Φιλήντα": "Φιλήντας",
    "Σκούφου": "Σκούφος",
    "Σκούφο": "Σκούφος",
    "Λούκαρη": "Λούκαρης",
    "Ροΐδη": "Ροΐδης",
    "Πάλλη": "Πάλλης",
    "Βλαστό": "Βλαστός",
    "Βλαστού": "Βλαστός",
    "Ψαλλίδα": "Ψαλλίδας",
    "Παλαμά": "Παλαμάς",
    "Δελμούζο": "Δελμούζος",
    "Δελμούζου": "Δελμούζος",
    "Βούλγαρη": "Βούλγαρης",
    "Σοφιανού": "Σοφιανός",
    "Τυπάλδο": "Τυπάλδος",
    "Τρικούπη": "Τρικούπης",
    "Ψυχάρη": "Ψυχάρης",
    "Καταρτζή": "Καταρτζής",
    "Κοραή": "Κοραής",
    "Γαβριηλίδη": "Γαβριηλίδης",
    "Βηλαρά": "Βηλαράς",
    "Κωνσταντά": "Κωνσταντάς",
    "Σούτσο": "Σούτσος",
    "Σούτσου": "Σούτσος",
    "Βαλαωρίτη": "Βαλαωρίτης",
    "Βενιζέλου": "Βενιζέλος",
    "Βενιζέλο": "Βενιζέλος",
    "Τσούντα": "Τσούντας",
    "Καβαλιώτη": "Καβαλιώτης",
    "Βαρβαγιάννη": "Βαρβαγιάννης",
    "Γαζή": "Γαζής",
    "Μιστριώτη": "Μιστριώτης",
    "Χατζηδάκη": "Χατζηδάκης",
    "Κονδυλάκη": "Κονδυλάκης",
    "Κούμα": "Κούμας",
    "Μάξιμο": "Μάξιμος",
    "Οικονόμου": "Οικονόμος",
    "Πολυλά": "Πολυλάς",
    "Μηνιάτη": "Μηνιάτης",
    "Ραγκαβή": "Ραγκαβής",
    "Κόντου": "Κόντος",
    "Κόντο": "Κόντος",
    "Λασκαράτο": "Λασκαράτος",
    "Κομμητά": "Κομμητάς",
    "Χριστόπουλο": "Χριστόπουλος",
    "Βερναρδάκη": "Βερναρδάκης",
    "Φωτιάδη": "Φωτιάδης",
    "Σολωμού": "Σολωμός",
    "Βελεστινλή": "Βελεστινλής",
    "Ασωπίου": "Ασώπιος",
    "Χατζή": "Χατζής",
    "Δημητρακόπουλο": "Δημητρακόπουλος",
    "Λογοθέτη": "Λογοθέτης",
    "Παπαδόπουλο": "Παπαδόπουλος",
    "Οικονόμων": "Οικονόμος",
    "Πτωχοπρόδρομου": "Πτωχοπρόδρομος",
    "Πτωχοπρόδρομο": "Πτωχοπρόδρομος",
    "Πτωχοπρόδρομων": "Πτωχοπρόδρομος",
    "Τερτσέτη": "Τερτσέτης",
    "Γεωργίου": "Γεώργιος",
    "Ξενοφώντα": "Ξενοφών",
    "Ιλαρίωνα": "Ιλαρίων",
    "Φαρμακίδη": "Φαρμακίδης",
}

In [ ]:
combined_df["Scholars_Norm"] = (
    combined_df["Scholar"]
    .replace(name_mapping)
)

In [ ]:
#check
combined_df.head()

In [ ]:
# save
combined_df.to_csv('kordatos_quotes_framing.csv', index=False, encoding='utf-8')  
